In [ ]:
%load_ext autoreload
%autoreload 2

## Load Raw

In [ ]:
from smlm.io import load_raw
from smlm import viz

# EMCCD 的 SIF —— em_gain 自动从头信息读出
src = r"/Volumes/SMILeSSD/SPARK/20261004/SPARK Cy5-SO3-antibody-638nm-16mW-0p1s-3000frames-fov14.sif"
raw = load_raw(src)
print(raw.data.shape, raw.n_frames, raw.em_gain, raw.exposure_time)
pixel_size=106.6 # nm/pixel
# sCMOS 的 TIF,每 5 帧取 1 帧
#raw = load_raw("cell01.tif", interval=5)
#print(raw.images.shape, raw.em_gain)   # em_gain 为 None

In [ ]:
# 接下来交给 camera 层转光子
from smlm.camera import EMCCD
cam = EMCCD(conv_factor=5.75, em_gain=raw.em_gain)
image = cam.process(raw, pixel_size=106.6)

In [ ]:
# 保存转化后的光子图像为 tif(与源文件同目录同名, 加 _photons 后缀)
import os
from smlm.io import save_tif
dst = os.path.splitext(src)[0] + "_photons.tif"
save_tif(image, dst)

## Preprocessing

In [ ]:
from smlm.imgprocess import Pipeline, Crop, SelectFrames
# 预处理: 裁剪 ROI + 选前 2000 帧
prep = Pipeline(Crop(y=(1, 256), x=(1, 256)),
                SelectFrames(stop=9999))
stack = prep(image)

In [ ]:
stack.shape

## Detect

In [ ]:
from smlm.localize import LocalMaxDetector
import numpy as np

detector=LocalMaxDetector(k_sigma=1.6, edge=5, f_em=np.sqrt(2),
                 bg_window=15, max_size=7, min_neighbors=2, verbose=True)
cand = detector.detect(stack)

In [ ]:
ax = viz.overlay_points(stack, cand, frames=4090, color='r', marker='o')

## Localize

In [ ]:
from smlm.localize import MLELocalizer
localizer = MLELocalizer(edge=5, F_EM=np.sqrt(2.0), use_pixel=False,
                 use_gradient=True, n_jobs=8, verbose=True)
locs = localizer.localize(stack,cand)

In [ ]:
from smlm.localize import filter_locs
from smlm.viz import plot_distributions
locs = filter_locs(locs)
plot_distributions(locs)

In [ ]:
from smlm.io import save_locs
from smlm.io.cache import data_fingerprint

# 格式与 run_cached 完全一致 -> <src 去掉扩展名>.locs.npz
save_locs(
    locs, src,
    method={'detector': detector, 'localizer': localizer},
    data_hash=data_fingerprint(stack),
)

## Detect+Localize

In [ ]:
from smlm.localize import ClassicalPipeline, LocalMaxDetector, MLELocalizer, filter_locs
from smlm.io import run_cached
import numpy as np
method = ClassicalPipeline(
    detector=LocalMaxDetector(k_sigma=3.8, edge=5, f_em=np.sqrt(2),
                 bg_window=15, max_size=7, min_neighbors=2, verbose=True),
    localizer = MLELocalizer(edge=5, F_EM=np.sqrt(2.0), use_pixel=False,
                 use_gradient=True, n_jobs=1, verbose=True),
)
locs = run_cached(method, stack, src)   # 首次计算并保存, 之后直接读盘

In [ ]:
from smlm.viz import plot_distributions
locs = filter_locs(locs)
plot_distributions(locs)

In [ ]:
ax = viz.overlay_points(stack, locs, frames=0, color='r', marker='x', size=30, linewidth = 0.5, unit='pixel',pixel_size=pixel_size)

## Drift Correction

In [ ]:
from smlm.drift import CloudDriftEstimator
from smlm.imgprocess import DriftCorrector     # 校正侧已移至 imgprocess

# 1) 估计(只需要定位)
uncertainty = np.median(locs.lp)
est = CloudDriftEstimator(frames_per_seg=10, uncertainty=uncertainty, do_stage2=True)
drift_trace = est.estimate(locs, n_frames=stack.n_frames)
#drift_trace = DriftTrace.zeros(n_frames = stack.n_frames, pixel_size=pixel_size)  # 测试零漂移占位

# 2) 校正图像 + 定位(带缓冲裁切, 与 MATLAB correctDrift 对齐)
corr = DriftCorrector(crop=(231, 231), edge=5)
corr_image, corr_locs, keep_idx = corr.apply(trace=drift_trace, image=stack, locs=locs)

In [ ]:
from smlm.viz import plot_drift
fig, ax = plot_drift(drift_trace, ex_time=raw.exposure_time,unit='nm',pixel_size=pixel_size) 


In [ ]:
ax = viz.overlay_points(corr_image, corr_locs, frames=np.arange(5000,8999), color='r', marker='x', size=30, linewidth = 0.5)

In [ ]:
from smlm.io import save_movie
import os

# 保存到"原 sif 文件所在目录", 文件名以 sif 名为前缀, 便于和源数据放一起
out_dir = os.path.dirname(src)
tag = os.path.splitext(os.path.basename(src))[0]
print("输出目录:", out_dir)

# ① 只要图像, 原速播放 (fps = 1/exposure_time)
save_movie(corr_image, os.path.join(out_dir, f"{tag}_corrected_1x.mp4"),
           exposure_time=raw.exposure_time, frames=slice(0, 500))

# ③ 图像 + 逐帧定位点叠加
save_movie(corr_image, os.path.join(out_dir, f"{tag}_overlay_1x.mp4"), frames=slice(0, 500),
           exposure_time=raw.exposure_time, speed=1, overlay=corr_locs,
           overlay_kw=dict(c='lime', marker='x', s=18, linewidths=0.5))


## Emitter Analysis

In [ ]:
from smlm.emitter import analyze_emitters

emitters, emitters_raw = analyze_emitters(
    corr_locs, n_frames=None,
    search_radius=1.0,
    max_gap=5000,
    min_frames=10,
    ecc_th=3.5,
    first_frame_range=(0,5000),
    motion='static',do_merge=False
)

In [ ]:
from smlm.emitter import collect_statistics
stats = collect_statistics(emitters, locs, raw.exposure_time, 0)


In [ ]:
from smlm.viz import plot_trajectories, plot_bleaching_summary, plot_emitter_stats
plot_trajectories(corr_image.data, emitters, linewidth=0.7, frame_idx=np.arange(0, 5000), dpi=250)
plot_emitter_stats(stats)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# 让图更容易看：
# 1) 左图保留“保守版有效不确定性”做颜色编码，适合看覆盖完整性和疑似问题 emitter
# 2) 右图改成“超分辨概率图”：把 emitter 作为带权重的 Gaussian 混合分布叠加成图像，
#    强调结构分布，不是看单个点；更适合作为可信的 SR 图像
# 3) 右图显式采用可信度加权，压低背景和高不确定性点，避免“噪声扩散”污染真正结构
# 4) 只保留 emitter centroid，避免重复闪烁叠加造成视觉模糊

render_factor = 8
# 可调参数：控制右图“背景 / 高不确定 emitter”的压低程度。
# 1) downweight_strength: 总压低强度，越大越压低
# 2) lp_cut_quantile: 超出该分位数的 emitter 被认定为高不确定，后续强烈压低
# 3) min_trust: 最低可信度门限，低于它的 emitter 直接丢掉
# 4) trusted_emitters_threshold: summary 输出里用到的可信 emitter 判断阈值
# 5) gmm_width_scale: 手动放大/缩小 GMM 的扩散宽度；如果 lp 偏小，可以调到 1.5~2.5
#
downweight_strength = 0
lp_cut_quantile = 100
min_trust = 1e-10
trusted_emitters_threshold = 0.15
gmm_width_scale = 1.5
n_emitters = len(emitters)
if n_emitters == 0:
    raise ValueError("No emitters available for SR rendering.")

h, w = corr_image.data.shape[1], corr_image.data.shape[2]
sr_h, sr_w = h * render_factor, w * render_factor

sr_equal_rgb = np.zeros((sr_h, sr_w, 3), dtype=np.float32)
prob_map = np.zeros((sr_h, sr_w), dtype=np.float32)

# 保守版：编号/计数修正很轻，避免让低计数 emitter 被过度放大
emitter_n_locs = np.asarray([em.n_locs for em in emitters], dtype=float)

if hasattr(corr_locs, "lp") and corr_locs.lp is not None:
    emitter_lp = np.asarray([
        np.median(corr_locs.lp[em.loc_idx]) if em.n_locs > 0 else np.nan
        for em in emitters
    ], dtype=float)
else:
    emitter_lp = np.full(n_emitters, np.nan, dtype=float)

if np.any(np.isfinite(emitter_n_locs)):
    n_count = np.clip(emitter_n_locs, 1.0, None)
    emitter_lp_eff = emitter_lp.copy()
    valid = np.isfinite(emitter_lp_eff)
    emitter_lp_eff[valid] = emitter_lp[valid] / np.sqrt(n_count[valid])
else:
    emitter_lp_eff = emitter_lp.copy()

finite_lp = np.isfinite(emitter_lp_eff)
if np.any(finite_lp):
    lp_ref = float(np.nanmedian(emitter_lp_eff[finite_lp]))
    lp_ref = max(lp_ref, 1e-6)
    lp_min = float(np.nanpercentile(emitter_lp_eff[finite_lp], 5))
    lp_max = float(np.nanpercentile(emitter_lp_eff[finite_lp], 95))
    lp_min = max(lp_min, 1e-3)
    lp_max = max(lp_max, lp_min + 1e-3)
else:
    lp_ref = 20.0
    lp_min, lp_max = 0.0, 1.0

# 左图：点状高对比渲染，但用保守有效不确定性给颜色
for i, em in enumerate(emitters):
    if em.n_locs <= 0:
        continue

    x0 = float(np.mean(corr_locs.x[em.loc_idx]))
    y0 = float(np.mean(corr_locs.y[em.loc_idx]))
    lp_nm = float(emitter_lp_eff[i]) if np.isfinite(emitter_lp_eff[i]) else lp_ref

    t = np.clip((lp_nm - lp_min) / max(lp_max - lp_min, 1e-6), 0.0, 1.0)
    color = np.array([
        0.10 + 0.90 * t,
        0.80 * (1.0 - 0.75 * t),
        0.18 + 0.12 * (1.0 - t)
    ], dtype=np.float32)

    x_sr = x0 * render_factor
    y_sr = y0 * render_factor

    radius = 1
    y_min = max(0, int(np.floor(y_sr)) - radius)
    y_max = min(sr_h, int(np.ceil(y_sr)) + radius + 1)
    x_min = max(0, int(np.floor(x_sr)) - radius)
    x_max = min(sr_w, int(np.ceil(x_sr)) + radius + 1)
    yy, xx = np.mgrid[y_min:y_max, x_min:x_max]
    dist2 = (xx - x_sr) ** 2 + (yy - y_sr) ** 2
    patch = np.exp(-dist2 / (2.0 * 0.9 ** 2))
    patch = np.clip(patch * 2.0, 0.0, 1.0)
    sr_equal_rgb[y_min:y_max, x_min:x_max] = np.maximum(
        sr_equal_rgb[y_min:y_max, x_min:x_max],
        patch[..., None] * color[None, None, :]
    )

# 右图：真正的超分辨概率图渲染：把 emitter 当作带权重的 Gaussian 混合分布叠加成图像
# 这是“概率分布视角”的 SR 重建，不是单纯散点扩散；高不确定性和背景点被显著下权重
raw_lp = emitter_lp.copy()
finite_raw = np.isfinite(raw_lp)
if np.any(finite_raw):
    lp_raw_ref = float(np.nanmedian(raw_lp[finite_raw]))
    lp_raw_ref = max(lp_raw_ref, 1e-6)
else:
    lp_raw_ref = 20.0

# 可信度建模：低不确定性 + 足够定位次数 -> 更可信；高不确定性、背景点 -> 强烈压低
emitter_trust = np.zeros(n_emitters, dtype=np.float32)
valid = np.isfinite(emitter_lp_eff)
if np.any(valid):
    lp_eff_valid = emitter_lp_eff[valid]
    count_valid = emitter_n_locs[valid]
    count_ref = float(np.nanmedian(count_valid)) if np.any(np.isfinite(count_valid)) else 1.0
    count_ref = max(count_ref, 1.0)

    if len(lp_eff_valid) > 1:
        lp_cut = float(np.nanpercentile(lp_eff_valid, lp_cut_quantile))
    else:
        lp_cut = float(np.max(lp_eff_valid))

    trust = np.clip(1.0 - (lp_eff_valid - lp_min) / max(lp_max - lp_min, 1e-6), 0.0, 1.0)
    trust *= np.clip(np.sqrt(count_valid / count_ref), 0.5, 2.0)

    # 可调压低强度：
    # downweight_strength = 0.0 -> 不压低
    # downweight_strength = 1.0 -> 中等压低
    # downweight_strength = 2.0 -> 更强压低
    low_confidence_weight = 0.15 / max(downweight_strength, 1e-3)
    trust *= np.where(lp_eff_valid <= lp_cut, 1.0, low_confidence_weight)
    emitter_trust[valid] = trust

if np.any(emitter_trust > 0):
    emitter_trust = np.clip(emitter_trust, 0.0, 1.0)
    emitter_trust = emitter_trust / max(np.nanmax(emitter_trust), 1e-6)

for i, em in enumerate(emitters):
    if em.n_locs <= 0:
        continue
    if emitter_trust[i] <= min_trust:
        continue

    x0 = float(np.mean(corr_locs.x[em.loc_idx]))
    y0 = float(np.mean(corr_locs.y[em.loc_idx]))
    lp_raw = float(raw_lp[i]) if np.isfinite(raw_lp[i]) else lp_raw_ref
    weight = float(emitter_trust[i])

    # gmm_width_scale 可手动调大/调小；如果 lp 偏小导致 Gaussian 过窄，可调到 1.5~2.5
    sigma_img = gmm_width_scale * (0.8 + 1.1 * (lp_raw / max(lp_raw_ref, 1e-6))) * (0.55 + 0.75 * weight)
    sigma_img = np.clip(sigma_img, 0.8, 6.0)
    amp_img = (0.8 + 0.5 * (lp_raw / max(lp_raw_ref, 1e-6))) * weight

    x_sr = x0 * render_factor
    y_sr = y0 * render_factor
    radius_img = int(np.ceil(4.0 * sigma_img))
    y_min = max(0, int(np.floor(y_sr)) - radius_img)
    y_max = min(sr_h, int(np.ceil(y_sr)) + radius_img + 1)
    x_min = max(0, int(np.floor(x_sr)) - radius_img)
    x_max = min(sr_w, int(np.ceil(x_sr)) + radius_img + 1)
    yy, xx = np.mgrid[y_min:y_max, x_min:x_max]
    patch_img = np.exp(-((xx - x_sr) ** 2 + (yy - y_sr) ** 2) / (2.0 * sigma_img ** 2))
    prob_map[y_min:y_max, x_min:x_max] += amp_img * patch_img

if prob_map.max() > 0:
    prob_map /= prob_map.max()
    prob_map = np.clip(prob_map, 0.0, 1.0)
    prob_map = prob_map ** 0.85

if sr_equal_rgb.max() > 0:
    sr_equal_rgb /= sr_equal_rgb.max()
    sr_equal_rgb = np.clip(sr_equal_rgb, 0.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(17, 7), dpi=180, gridspec_kw={'wspace': 0.35})
for ax in axes:
    ax.set_aspect('equal')
    ax.set_facecolor('black')

ax = axes[0]
ax.imshow(sr_equal_rgb, origin='upper', interpolation='nearest',
          extent=[0, sr_w, sr_h, 0], vmin=0.0, vmax=1.0)
ax.set_title('Equal brightness per emitter\ncolor = effective localization uncertainty')
ax.set_xlabel('x (SR pixel)')
ax.set_ylabel('y (SR pixel)')

cmap_redgreen = LinearSegmentedColormap.from_list(
    'red_green', ['#2ecc71', '#f4d03f', '#e74c3c']
)
sm = plt.cm.ScalarMappable(cmap=cmap_redgreen, norm=plt.Normalize(vmin=lp_min, vmax=lp_max))
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.04)
cb.ax.set_facecolor('black')
cb.ax.set_title('Effective uncertainty', color='white', fontsize=9, pad=8)
cb.set_label('Effective localization uncertainty (nm)')

ax = axes[1]
img_lp = np.clip(prob_map, 0.0, None)
vmin, vmax = np.percentile(img_lp[img_lp > 0], (1, 99.5)) if np.any(img_lp > 0) else (0.0, 1.0)
if vmax <= vmin:
    vmin, vmax = 0.0, 1.0
ax.imshow(img_lp, cmap='magma', origin='upper', interpolation='nearest',
          extent=[0, sr_w, sr_h, 0], vmin=vmin, vmax=vmax)
ax.set_title(
    'GMM probability map\n'
    f'width×{gmm_width_scale:g}, downweight={downweight_strength}, '
    f'lp_cut={lp_cut_quantile:g}, min_trust={min_trust:g}, '
    f'trusted={int(np.sum(emitter_trust > trusted_emitters_threshold))}'
)
ax.set_xlabel('x (SR pixel)')
ax.set_ylabel('y (SR pixel)')

sr_img_equal = sr_equal_rgb
sr_img_lp = img_lp

## Trace Extraction

In [ ]:
from smlm.trace import ApertureTraceExtractor
extractor = ApertureTraceExtractor(edge=3, do_bg_subtract=True, bg_edge=6, excl_radius=4, sigma_clip_n=3)

In [ ]:
traces = extractor.extract(emitters, corr_image.data, corr_image.n_frames)

In [ ]:
from smlm.viz import check_trace
for i in range(1,40):
    # photons_i 长度 == emitters[i].frame.size(每个 localization 一个值)
    photons_i = corr_locs.photons[emitters[i].loc_idx]     # 按你的 Localizations 取法
    check_trace(traces[i], emitter=emitters[i], ex_time=raw.exposure_time,
                mle_brightness=photons_i, use_time=True, show_raw=False, show_track=False, panels='trace', frame_range=(0,1500))

## Emitter Window Movies（挑选单分子）

把若干 emitter 的**窗口小视频**并排播出来，人工判断哪些是可信的单分子
（亮度稳定、单步漂白、无并发/邻分子串扰），再挑序号导出成 mp4。

- 窗口以该 emitter 定位的**中位数**为中心，尺寸 `(2*half+1)^2`
- 红色 `×` 是该分子自己的定位点（`corr_locs[emitters[i].loc_idx]`）
- 播放是内联动图，默认自动抽帧到 `max_frames` 帧以内以控制体积


In [ ]:
from smlm.viz import show_emitter_movies

# 看前 24 个 emitter 的小窗视频(窗口 15x15, 取前 200 帧, 每行 6 个)
show_emitter_movies(corr_image, emitters, idx=[4,10], half=7,
                    frames=slice(0, 1500), overlay=True, ncols=6)


In [ ]:
from smlm.viz import emitter_roi
from smlm.io import save_movie
import os

# ---- 把上面挑好的序号填这里 ----
picked = [4,10]

out_dir = os.path.dirname(src)
tag = os.path.splitext(os.path.basename(src))[0]

for i in picked:
    # 以该分子为中心的窗口(原片前 1600 帧)
    roi = emitter_roi(corr_image, emitters[i], half=7, frames=slice(0, 1600))

    # save_movie(data, path, ...): 前两个是位置参数, 其余一律用关键字
    save_movie(
        roi,                                              # data (位置参数 1)
        os.path.join(out_dir, f"{tag}_em{i:03d}.mp4"),    # path (位置参数 2)
        exposure_time=raw.exposure_time, speed=5,
        overlay=corr_locs[emitters[i].loc_idx],
        trace=traces[i], clim=(15,85),
    )

## Trace Analysis

In [ ]:
from smlm.trace import get_analyzer, analyze_traces, plot_state_gallery
specs = get_analyzer("changepoint").param_specs()
for name, (default, help_) in specs.items():
    print(f"{name:18s} = {default!r:12}  {help_}")

In [ ]:
results = analyze_traces(traces, method="changepoint",penalty=3, bleach_tail = 50)

In [ ]:
fig = plot_state_gallery(results, n_max=20)

In [ ]:
from smlm.trace import plot_summary
fig = plot_summary(results)